# Stability versus $1-\rho$

This experiment complements the earlier batch-size stability scan. It fixes the batch size and scans the momentum damping $g=1-\rho$ for **SGD, Polyak, and Nesterov**. For every $(algorithm,g,\eta)$ and random seed, training is declared stable when the final moving-average population excess loss remains finite and below the configured threshold.

The expected boundaries are

$$\eta_{\rm SGD}^{\rm crit}\asymp 1,\qquad \eta_{\rm Polyak}^{\rm crit}\asymp\min\{1,B(1-\rho)\},\qquad \eta_{\rm Nesterov}^{\rm crit}\asymp\min\{1,B^\beta(1-\rho)\}.$$

Thus both momentum curves should be linear in $1-\rho$ before reaching their deterministic ceilings, with Nesterov shifted upward by its stronger $B^\beta$ dependence.

In [ ]:
from pathlib import Path
import json, math, platform, time

import matplotlib.pyplot as plt
import numpy as np
import torch

if torch.cuda.is_available():
    DEVICE, DTYPE = torch.device('cuda'), torch.float64
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = torch.device('mps'), torch.float32
else:
    DEVICE, DTYPE = torch.device('cpu'), torch.float64

OUTPUT_DIR = Path('results/new_stability')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'device={DEVICE}, dtype={DTYPE}, torch={torch.__version__}, numpy={np.__version__}')

## Experiment setting

The PLK problem and the stability rule match the earlier experiment: $\beta=2$, $s=3$, no label noise, dimension 4096, fresh exact Gaussian mini-batches, 2000 steps, five seeds, and a final-window loss threshold of one. The default configuration is intended for a CUDA GPU. Set `PAPER_RUN=False` only for a short smoke test.

In [ ]:
PAPER_RUN = True
BETA, S, SIGMA = 2.0, 3.0, 0.0
BATCH_SIZE = 32
if PAPER_RUN:
    DIM = 4096
    ONE_MINUS_RHO_GRID = 2.0 ** np.arange(-24, 0, 2, dtype=float)
    ETA_GRID = np.geomspace(2.0 ** -22, 8.0, 50)
    STEPS, WINDOW, N_SEEDS, CHUNK = 2000, 20, 5, 2048
else:
    DIM = 256
    ONE_MINUS_RHO_GRID = 2.0 ** np.arange(-10, 0, 2, dtype=float)
    ETA_GRID = np.geomspace(2.0 ** -10, 4.0, 20)
    STEPS, WINDOW, N_SEEDS, CHUNK = 100, 20, 1, 256
STABLE_THRESHOLD, DIVERGENCE_LOSS = 1.0, 1e12
SEED = 20260822
ALGORITHMS = ('sgd', 'polyak', 'nesterov')
DISPLAY = {'sgd': 'SGD', 'polyak': 'Polyak', 'nesterov': 'Nesterov'}

config = {
    'preset': 'paper' if PAPER_RUN else 'quick', 'beta': BETA, 's': S, 'sigma': SIGMA,
    'dim': DIM, 'batch_size': BATCH_SIZE,
    'one_minus_rho_grid': ONE_MINUS_RHO_GRID.tolist(), 'eta_grid': ETA_GRID.tolist(),
    'steps': STEPS, 'window': WINDOW, 'stable_threshold': STABLE_THRESHOLD,
    'divergence_loss': DIVERGENCE_LOSS, 'n_seeds': N_SEEDS, 'seed': SEED,
    'device': str(DEVICE), 'dtype': str(DTYPE), 'python': platform.python_version(),
    'torch': torch.__version__, 'numpy': np.__version__,
}
print(json.dumps(config, indent=2))

In [ ]:
j = torch.arange(1, DIM + 1, device=DEVICE, dtype=DTYPE)
lam = j.pow(-BETA)
theta_star = j.pow(-0.5 * (1.0 + (S - 1.0) * BETA))
sqrt_lam = lam.sqrt()
initial_loss = float((0.5 * torch.sum(lam * theta_star.square())).item())
print(f'lambda range=[{lam[-1].item():.3e}, {lam[0].item():.3e}], initial loss={initial_loss:.6f}')

@torch.no_grad()
def exact_gradient(delta, batch_size, generator):
    grad = torch.zeros_like(delta)
    remaining = int(batch_size)
    while remaining:
        n = min(CHUNK, remaining)
        x = torch.randn((n, DIM), device=DEVICE, dtype=DTYPE, generator=generator)
        x.mul_(sqrt_lam)
        residual = x.mv(delta)
        if SIGMA:
            residual.sub_(torch.randn(n, device=DEVICE, dtype=DTYPE, generator=generator) * SIGMA)
        grad.add_(x.t().mv(residual))
        remaining -= n
    return grad / float(batch_size)

@torch.no_grad()
def run_trial(algorithm, eta, rho, seed):
    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(int(seed))
    delta = -theta_star.clone()
    previous = delta.clone()
    tail = []
    for step in range(1, STEPS + 1):
        momentum = delta - previous
        evaluation = delta + rho * momentum if algorithm == 'nesterov' else delta
        gradient = exact_gradient(evaluation, BATCH_SIZE, generator)
        new_delta = delta - eta * gradient
        if algorithm != 'sgd':
            new_delta.add_(rho * momentum)
        previous, delta = delta, new_delta
        loss = float((0.5 * torch.sum(lam * delta.square())).item())
        if not math.isfinite(loss) or loss > DIVERGENCE_LOSS:
            return False, math.inf, step
        tail.append(loss)
        if len(tail) > WINDOW:
            tail.pop(0)
    final_average = float(np.mean(tail))
    return final_average < STABLE_THRESHOLD, final_average, STEPS

print(run_trial('sgd', float(ETA_GRID[0]), 0.0, SEED))

## Run and cache the scan

SGD does not depend on $\rho$, so it is evaluated once per $(seed,\eta)$ and broadcast across the horizontal grid. Momentum trials use independent deterministic random streams. The cache contains every stability flag and final moving-average loss.

In [ ]:
cache_path = OUTPUT_DIR / 'new_stability_results.npz'
root_cache_path = Path('new_stability_results.npz')
if not cache_path.exists() and root_cache_path.exists():
    cache_path = root_cache_path
shape = (len(ALGORITHMS), N_SEEDS, len(ETA_GRID), len(ONE_MINUS_RHO_GRID))
if cache_path.exists():
    cached = np.load(cache_path, allow_pickle=False)
    cached_config = json.loads(str(cached['config']))
    if cached_config != config:
        print('Using the configuration and grids embedded in the existing cache.')
    config = cached_config
    ALGORITHMS = tuple(cached['algorithms'].tolist())
    ETA_GRID = cached['eta_grid'].copy()
    ONE_MINUS_RHO_GRID = cached['one_minus_rho_grid'].copy()
    BETA, S, SIGMA = float(config['beta']), float(config['s']), float(config['sigma'])
    BATCH_SIZE, N_SEEDS = int(config['batch_size']), int(config['n_seeds'])
    stable = cached['stable'].copy()
    final_average = cached['final_average'].copy()
    steps_done = cached['steps_done'].copy()
    completed = cached['completed'].copy()
    print(f'resuming from {cache_path}')
else:
    stable = np.zeros(shape, dtype=bool)
    final_average = np.full(shape, np.nan)
    steps_done = np.zeros(shape, dtype=np.int32)
    completed = np.zeros((len(ALGORITHMS), len(ONE_MINUS_RHO_GRID)), dtype=bool)

def save_cache():
    np.savez_compressed(
        cache_path, stable=stable, final_average=final_average, steps_done=steps_done,
        completed=completed, algorithms=np.asarray(ALGORITHMS), eta_grid=ETA_GRID,
        one_minus_rho_grid=ONE_MINUS_RHO_GRID, config=json.dumps(config, sort_keys=True),
    )

t0 = time.time()

for a, algorithm in enumerate(ALGORITHMS):
    rho_indices = [0] if algorithm == 'sgd' else range(len(ONE_MINUS_RHO_GRID))
    print(f'=== {DISPLAY[algorithm]} ===', flush=True)
    for g_idx in rho_indices:
        if completed[a, g_idx]:
            print(f'  skipping completed index {g_idx}', flush=True)
            continue
        g = float(ONE_MINUS_RHO_GRID[g_idx])
        rho = 0.0 if algorithm == 'sgd' else 1.0 - g
        print(f'  1-rho={g:.6g}', flush=True)
        for seed_idx in range(N_SEEDS):
            for eta_idx, eta in enumerate(ETA_GRID):
                trial_seed = SEED + a * 10_000_000 + g_idx * 100_000 + seed_idx * 1_000 + eta_idx
                out = run_trial(algorithm, float(eta), rho, trial_seed)
                stable[a, seed_idx, eta_idx, g_idx] = out[0]
                final_average[a, seed_idx, eta_idx, g_idx] = out[1]
                steps_done[a, seed_idx, eta_idx, g_idx] = out[2]
        completed[a, g_idx] = True
        save_cache()
    if algorithm == 'sgd':
        stable[a] = np.repeat(stable[a, :, :, :1], len(ONE_MINUS_RHO_GRID), axis=2)
        final_average[a] = np.repeat(final_average[a, :, :, :1], len(ONE_MINUS_RHO_GRID), axis=2)
        steps_done[a] = np.repeat(steps_done[a, :, :, :1], len(ONE_MINUS_RHO_GRID), axis=2)
        completed[a, :] = True
        save_cache()

elapsed = time.time() - t0
save_cache()
with (OUTPUT_DIR / 'new_stability_config.json').open('w') as f:
    json.dump(config | {'elapsed_seconds': elapsed}, f, indent=2)
print(f'finished in {elapsed:.1f}s; cache saved to {OUTPUT_DIR.resolve()}')

In [ ]:
per_seed_boundary = np.full((len(ALGORITHMS), N_SEEDS, len(ONE_MINUS_RHO_GRID)), np.nan)
for a in range(len(ALGORITHMS)):
    for seed_idx in range(N_SEEDS):
        for g_idx in range(len(ONE_MINUS_RHO_GRID)):
            indices = np.flatnonzero(stable[a, seed_idx, :, g_idx])
            if indices.size:
                per_seed_boundary[a, seed_idx, g_idx] = ETA_GRID[indices[-1]]
boundary = np.nanmean(per_seed_boundary, axis=1)
boundary_std = np.nanstd(per_seed_boundary, axis=1)

def fit_linear_one_kink(values, g, min_points_per_side=2, grid_size=4001):
    """Fit eta = min(level, coefficient * (1-rho)) with one kink.

    log(eta) = log(level) + min(log(g) - log(g_break), 0).
    The log--log slope is fixed to one, as required by linear scaling in g.
    """
    finite = np.isfinite(values) & (values > 0) & np.isfinite(g) & (g > 0)
    x, y = np.log(g[finite]), np.log(values[finite])
    order = np.argsort(x)
    x, y = x[order], y[order]
    if x.size < 2 * min_points_per_side:
        raise ValueError('Not enough finite points to fit both sides of one kink.')

    lo = x[min_points_per_side - 1]
    hi = x[-min_points_per_side]
    candidates = np.linspace(lo, hi, grid_size)
    best = None
    for x_break in candidates:
        hinge = np.minimum(x - x_break, 0.0)
        log_level = float(np.mean(y - hinge))
        residual = y - (log_level + hinge)
        sse = float(residual @ residual)
        if best is None or sse < best['sse_log']:
            best = {
                'coefficient': float(np.exp(log_level - x_break)),
                'level': float(np.exp(log_level)),
                'slope': 1.0,
                'breakpoint': float(np.exp(x_break)),
                'sse_log': sse,
            }
    if best is None:
        raise ValueError('The data do not support a linear-to-flat fit.')
    return best

fits = {}
for a, algorithm in enumerate(ALGORITHMS):
    values = boundary[a]
    if algorithm == 'sgd':
        level = float(np.exp(np.nanmean(np.log(values))))
        fits[algorithm] = {'level': level, 'slope': 0.0}
    else:
        fits[algorithm] = fit_linear_one_kink(values, ONE_MINUS_RHO_GRID)

np.savez_compressed(
    OUTPUT_DIR / 'new_stability_summary.npz', boundary=boundary, boundary_std=boundary_std,
    per_seed_boundary=per_seed_boundary, algorithms=np.asarray(ALGORITHMS),
    one_minus_rho_grid=ONE_MINUS_RHO_GRID, eta_grid=ETA_GRID, fits=json.dumps(fits),
)
with (OUTPUT_DIR / 'new_stability_summary.json').open('w') as f:
    json.dump({'config': config, 'fits': fits, 'boundary': {DISPLAY[x]: boundary[i].tolist() for i, x in enumerate(ALGORITHMS)}}, f, indent=2)
fits

In [ ]:
colors = {'sgd': 'tab:green', 'polyak': 'tab:blue', 'nesterov': 'tab:orange'}
markers = {'sgd': '^', 'polyak': 'o', 'nesterov': 's'}
plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.labelsize': 16,
    'axes.titlesize': 20,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 10,
})
fig, ax = plt.subplots(figsize=(5.0, 4.0))

for a, algorithm in enumerate(ALGORITHMS):
    values = boundary[a]
    mask = np.isfinite(values) & (values > 0)
    ax.plot(
        ONE_MINUS_RHO_GRID[mask], values[mask], linestyle='None',
        marker=markers[algorithm], markersize=6, color=colors[algorithm],
        markerfacecolor=colors[algorithm], markeredgecolor=colors[algorithm],
        markeredgewidth=1.1, label='_nolegend_', zorder=3,
    )
    if algorithm == 'sgd':
        prediction = np.full_like(ONE_MINUS_RHO_GRID, fits[algorithm]['level'])
        label = 'SGD Prediction'
    else:
        dense_g = np.geomspace(ONE_MINUS_RHO_GRID.min(), ONE_MINUS_RHO_GRID.max(), 500)
        prediction = np.minimum(
            fits[algorithm]['level'],
            fits[algorithm]['coefficient'] * dense_g,
        )
        plot_g = dense_g
        label = f'{DISPLAY[algorithm]} Prediction'
    if algorithm == 'sgd':
        plot_g = ONE_MINUS_RHO_GRID
    ax.plot(plot_g, prediction, '--', lw=1.8, color=colors[algorithm], label=label, zorder=2)

ax.set_xscale('log', base=2)
ax.set_yscale('log')
x_exponents = np.arange(
    int(np.ceil(np.log2(ONE_MINUS_RHO_GRID.min()))),
    int(np.floor(np.log2(ONE_MINUS_RHO_GRID.max()))) + 1,
)
ax.set_xlim(ONE_MINUS_RHO_GRID.min(), ONE_MINUS_RHO_GRID.max())
ax.set_xticks(2.0 ** x_exponents)
ax.set_xticklabels([rf'$2^{{{exponent}}}$' for exponent in x_exponents])
ax.set_ylim(1e-2, 2.0)
ax.set_xlabel(r'Momentum Damping  $1-\rho$')
ax.set_ylabel(r'Stability Boundary  $\eta$')
ax.set_title('Critical learning rates')
ax.grid(False)
ax.legend(loc='lower right', handlelength=2.8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'new_stability.png', dpi=180, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'new_stability.pdf', bbox_inches='tight')
plt.show()
for algorithm in ('polyak', 'nesterov'):
    print(f"{DISPLAY[algorithm]}: slope={fits[algorithm]['slope']:.3f}, "
          f"breakpoint={fits[algorithm]['breakpoint']:.6g}, "
          f"level={fits[algorithm]['level']:.6g}")

## Reading the result

Polyak and Nesterov are fitted directly to $\eta^{\rm crit}=\min\{L,c(1-\rho)\}$. Thus the rising branch is constrained to be linear in $1-\rho$ (unit slope in log--log coordinates), and continuity with the horizontal level $L$ guarantees exactly one kink. The fitted `breakpoint=L/c` is the transition value of $1-\rho$, and `level=L` is the deterministic stability ceiling. SGD provides the $\rho$-independent control.